# Plan-and-Execute

복잡한 작업을 단계별로 계획하고 실행하며, 중간 결과에 따라 계획을 수정하는 Plan-and-Execute 패턴을 배운다.

In [ ]:
import operator
from typing import Annotated, TypedDict, Union

from dotenv import load_dotenv
from IPython.display import Image, display
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field


load_dotenv()

MODEL_NAME = "gemini-3.6-flash"

### Plan-and-Execute란

Plan-and-Execute는 먼저 전체 계획을 세운 뒤 각 단계를 실행하는 패턴이다. 실행 결과를 검토하고 필요에 따라 남은 계획을 수정할 수 있다.

| 구분 | ReAct | Plan-and-Execute |
|---|---|---|
| 계획 범위 | 매 순간 다음 행동 결정 | 전체 계획을 먼저 수립 |
| 장점 | 단순하고 빠름 | 복잡한 목표와 제약 조건 처리에 유리 |
| 단점 | 전체 흐름을 놓칠 수 있음 | 계획과 재계획 비용 발생 |
| 적합한 작업 | 단일 조회, 간단한 질의 | 조사, 비교, 일정 설계, 보고서 작성 |

```text
입력 → Planner → Executor → Replanner → Executor → ... → 최종 응답
```

Planner는 실행할 단계 목록을 만들고, Executor는 현재 계획의 첫 단계를 수행한다. Replanner는 실행 결과가 충분한지 판단하여 종료하거나 남은 계획을 교체한다.

### State와 출력 모델 정의

Plan-and-Execute에서는 현재 계획뿐 아니라 완료된 단계와 최종 응답도 추적해야 한다. Planner와 Replanner의 출력은 Pydantic 모델로 정의한다.

| 필드 | 타입 | 역할 |
|---|---|---|
| `input` | `str` | 사용자의 원래 요청 |
| `plan` | `list[str]` | 앞으로 실행할 단계 |
| `past_steps` | `list[tuple[str, str]]` | `(실행한 단계, 실행 결과)` 튜플의 목록 |
| `response` | `str` | 사용자에게 전달할 최종 응답 |

`past_steps`는 실행 이력을 보존해야 하므로 `operator.add` reducer로 누적한다. `plan`은 Replanner가 남은 계획 전체를 바꿀 수 있도록 reducer 없이 덮어쓴다.

In [ ]:
llm = ChatGoogleGenerativeAI(model=MODEL_NAME)

class PlanExecuteState(TypedDict):
    input: str
    plan: list[str]
    past_steps: Annotated[list[tuple[str, str]], operator.add]
    response: str

class Plan(BaseModel):
    """앞으로 수행할 계획"""

    steps: list[str] = Field(description="순서대로 수행할 단계")


class Response(BaseModel):
    """사용자에게 전달할 최종 응답"""

    response: str = Field(description="검색 결과를 반영한 최종 여행 계획")


class ReplanDecision(BaseModel):
    """Replanner가 선택한 다음 결정"""

    decision: Union[Plan, Response]

    # 따로 두지 않는 이유는 두 응답이 다 나올 수 있기 때문에
    # plan : Plan
    # response : Response

### Planner

Planner는 사용자 요청을 구체적이고 실행 가능한 검색 단계로 나눈다. 한 단계에는 하나의 조사 목표만 포함한다.

In [ ]:
planner_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """당신은 여행 요청을 실행 계획으로 만드는 Planner입니다.
각 단계는 Google Search로 조사할 수 있는 구체적인 작업이어야 합니다.
교통, 숙박, 관광지 운영 정보, 식당, 예산 검증처럼 서로 다른 조사 목표를 나누세요.
실행하지 않은 결과를 미리 판단하지 마세요.
계획은 최대 5단계로 작성하세요.""",
    ),
    ("human", "{input}"),
])

planner = planner_prompt | llm.with_structured_output(Plan)


def plan_step(state: PlanExecuteState):
    """사용자 요청을 바탕으로 초기 계획을 생성한다."""
    result = planner.invoke({"input": state["input"]})
    return {"plan": result.steps}

### Executor

Executor는 현재 계획의 첫 단계를 수행한다. Google Search가 연결된 Gemini를 사용해 최신 여행 정보를 조사하고 출처 URL과 함께 결과를 정리한다.

In [ ]:
search_llm = llm.bind_tools([{"google_search": {}}])

def execute_step(state: PlanExecuteState):
    """현재 계획의 첫 단계를 Google Search로 실행한다."""
    current_step = state["plan"][0]

    past_context = ""
    if state.get("past_steps"):
        past_context = "\n\n지금까지 확인한 결과:\n" + "\n".join(
            f"- {step}: {result}"
            for step, result in state["past_steps"]
        )

    task = f"""다음 여행 조사 단계를 수행하세요.

현재 단계:
{current_step}

요구사항:
- Google Search로 최신 정보를 확인하세요.
- 가격, 운영시간, 이동시간을 구체적으로 정리하세요.
- 확인한 정보의 출처 URL을 포함하세요.
- 현재 단계에 필요한 내용만 간결하게 작성하세요.
{past_context}"""

    # ReAct Agent
    result = search_llm.invoke(task)

    return {"past_steps": [(current_step, result.text)]}

### Replanner

Replanner는 원래 요청과 지금까지의 검색 결과를 비교한다. 정보가 부족하면 완료된 단계를 제외한 새 계획을 만들고, 조건을 판단할 수 있으면 최종 여행 계획을 작성한다.

Replanner는 매 실행 이후 `past_steps`를 확인한다.

| 상황 | 판단 |
|---|---|
| 필요한 정보가 부족함 | 추가 조사 단계를 계획에 포함 |
| 검색 결과가 조건에 맞지 않음 | 대안을 조사하도록 계획 수정 |
| 모든 조건을 판단할 수 있음 | 최종 응답 생성 |

`Union[A, B]`는 값의 타입으로 A 또는 B 중 하나를 허용한다는 뜻이다. 예를 들어 `Union[Plan, Response]`로 선언하면 `Plan` 객체나 `Response` 객체 중 하나를 저장할 수 있다.

Replanner는 한 번의 호출에서 **남은 계획을 반환하거나 최종 응답을 반환한다**. 두 결과가 동시에 필요한 것은 아니므로 `ReplanDecision`의 `decision` 필드를 `Union[Plan, Response]`로 정의해 둘 중 하나만 선택하게 한다. 반면 그래프의 State에서는 실행할 `plan`과 종료 시 사용할 `response`의 역할이 다르므로 별도 필드로 관리한다.

완료된 단계를 새 계획에서 제거하지 않으면 같은 검색이 반복될 수 있다. 프롬프트에서 완료된 단계의 반복을 금지하고 `recursion_limit`으로 무한 반복을 방지한다. `recursion_limit`은 한 번의 그래프 실행에서 허용하는 최대 노드 실행 횟수다. 같은 노드를 반복해서 실행하는 경우도 매번 횟수에 포함된다. 아래 실행 코드의 `recursion_limit=20`은 노드 실행을 최대 20번까지 허용한다는 뜻이며, 한도를 초과하면 `GraphRecursionError`가 발생한다.

In [ ]:
replanner_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """당신은 여행 계획의 진행 상황을 검토하는 Replanner입니다.

원래 요청과 완료한 조사 결과를 보고 다음 중 하나를 선택하세요.
- 필요한 정보가 부족하면 완료한 단계를 반복하지 말고 남은 Plan을 반환합니다.
- 조건을 충족하는 여행 계획을 작성할 수 있으면 Response를 반환합니다.
- 검색 결과에서 제약 조건을 만족하지 못한 항목이 발견되면 대안을 조사하는 단계를 추가합니다.
- 최종 응답에는 일정, 예상 비용, 이동 방법, 주의사항, 출처를 포함합니다.""",
    ),
    (
        "human",
        """목표:
{input}

현재 계획:
{plan}

완료한 단계:
{past_steps}""",
    ),
])

replanner = replanner_prompt | llm.with_structured_output(ReplanDecision)



def replan_step(state: PlanExecuteState):
    """검색 결과를 바탕으로 계획을 수정하거나 최종 응답을 생성한다."""
    result = replanner.invoke({
        "input": state["input"],
        "plan": "\n".join(
            f"- {step}"
            for step in state["plan"]
        ),
        "past_steps": "\n".join(
            f"- {step}: {step_result}"
            for step, step_result in state["past_steps"]
        ),
    })

    if isinstance(result.decision, Response):
        return {"response": result.decision.response}

    return {"plan": result.decision.steps}


def should_continue(state: PlanExecuteState):
    """최종 응답이 있으면 종료하고, 없으면 다음 단계를 실행한다."""

    # if state.get("response"):
    #     value = 'end'
    # else:
    #     value = 'continue'
    # return value

    value = "end" if state.get("response") else "continue" 
    return value

### 그래프 구성

Planner가 초기 계획을 만들고, Executor와 Replanner가 최종 응답이 생성될 때까지 반복한다.

In [ ]:
graph_builder = StateGraph(PlanExecuteState)

graph_builder.add_node("planner", plan_step)
graph_builder.add_node("executor", execute_step)
graph_builder.add_node("replanner", replan_step)

graph_builder.add_edge(START, "planner")
graph_builder.add_edge("planner", "executor")
graph_builder.add_edge("executor", "replanner")
graph_builder.add_conditional_edges(
    "replanner",
    should_continue,
    {
        "continue": "executor",
        "end": END,
    },
)

graph = graph_builder.compile()

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

### 실행

여러 조건을 함께 고려해야 하는 여행 계획을 요청한다. 검색 결과에서 휴무, 매진, 예산 초과 같은 문제가 확인되면 Replanner가 대안을 조사하도록 계획을 수정한다.

In [ ]:
inputs = {
    "input": (
        "다음 주 토요일에 서울에서 출발하는 강릉 당일치기 여행 계획을 만들어줘. "
        "대중교통을 중심으로 이동하고, 관광지 2곳 이상과 현지 식당을 포함해줘. "
        "1인 총예산은 15만 원 이하야. 운영시간과 예상 비용도 알려줘."
    )
}

config = {"recursion_limit": 20}

state_history = []

for mode, event in graph.stream(
    inputs,
    config=config,
    stream_mode=["updates", "values"],
):
    if mode == "values":
        state_history.append(event)
        continue

    for node_name, value in event.items():
        if node_name == "planner":
            print("\n[초기 계획]")
            for step in value["plan"]:
                print(f"- {step}")

        elif node_name == "executor":
            step, result = value["past_steps"][-1]
            print(f"\n[실행] {step}")
            print(result)

        elif node_name == "replanner" and value.get("plan"):
            print("\n[수정된 계획]")
            for step in value["plan"]:
                print(f"- {step}")

        elif node_name == "replanner" and value.get("response"):
            print("\n[최종 응답]")
            print(value["response"])

In [ ]:
print("\n[State 흐름]")

for index, state in enumerate(state_history):
    print(f"\n[State {index}]")

    for key, value in state.items():
        if key == "past_steps":
            print("past_steps:")
            for step, _ in value:
                print(f"  - {step}")
        elif key == "response":
            print("response:", bool(value))
            print(value)
        else:
            print(f"{key}:", value)

### 실습 문제

2026년 생성형 AI 트렌드를 조사하는 Plan-and-Execute 그래프를 처음부터 구현해보자.

사용자 요청:

> 2026년 생성형 AI 트렌드를 조사하고, 주요 트렌드 5개를 선정해 근거와 함께 요약 보고서를 작성해줘.


### 추가 실습: Replanner 없는 Plan-and-Execute

Replanner는 Plan-and-Execute의 필수 구성요소가 아니다. 초기 계획을 수정할 필요가 없다면 Planner가 만든 계획을 그대로 실행할 수 있다.

앞에서 구현한 여행 계획 그래프에서 Replanner를 제거하고, Planner가 처음 만든 계획을 순서대로 모두 실행하도록 수정해보자.

> 힌트: Executor가 현재 단계를 실행한 뒤 `{"plan": state["plan"][1:]}`을 반환하면 완료한 단계를 계획에서 제거할 수 있다. `plan`이 비었는지를 기준으로 Executor와 최종 응답 생성 노드 사이를 분기할 수 있다.